# NLP_01 — Des mots à une baseline évaluée

*Conception pédagogique : Mathieu Weill. Usage personnel d’apprentissage ; droits et crédits tiers conservés.*

**Compétition de travail : [tweets](https://www.kaggle.com/competitions/nlp-getting-started)**

**Question : Un modèle comprend-il le texte ou exploite-t-il des indices faciles ?**

7 heures pédagogiques, hors pauses. Les exemples sont des jalons techniques courts ; l’essentiel du temps est consacré aux exercices, revues et expériences. **Version étudiant.**

**[Repère de compréhension : Représenter le texte avec une référence simple](../../../course/chapitres/NLP_01/README.md)**


## Préparation et limites

Lire README.md, compétitions et règles. Les données officielles sont téléchargées personnellement et placées dans data/competitions/<slug>/. Aucun remplacement silencieux par des données simulées. Poids préentraînés à préparer avant NLP_02/CV_01/CV_02. Les limites de calcul et d’échanges dépendent du concours et du compte.

## Notions du jour

Disaster Tweets : une ligne = un tweet anglais, cible 1 si le texte décrit une catastrophe réelle, 0 sinon. La métrique officielle est le **F1 positif**, pas l’accuracy ni le F1 macro. La soumission contient `id,target`, avec des classes 0/1.

TF-IDF apprend un vocabulaire sur le train ; la régression logistique transforme ce vecteur en probabilité. Le seuil transforme la probabilité en décision. Choisir ce seuil sur OOF/validation, jamais sur les labels du test.

Les textes identiques normalisés restent dans le même ensemble. Auditer aussi les quasi-doublons et les événements : ce split ne garantit pas une généralisation chronologique. Certains tweets peuvent contenir du langage offensant.

Ce Getting Started a un classement roulant, sans prix en argent ni médailles. Son leaderboard public ne doit pas être présenté comme un test privé final distinct.

In [ ]:
from pathlib import Path
import sys,os,json,time
import numpy as np,pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
HERE=Path.cwd()
ROOT=next((p if (p/'mllab/competition.py').exists() else p/'student') for p in [HERE,*HERE.parents] if (p/'mllab/competition.py').exists() or (p/'student/mllab/competition.py').exists())
sys.path.insert(0,str(ROOT))
from mllab.competition import SPECS,read_competition,protocol,run_competition,binary_scores,get_folder
KEY='tweets';SPEC=SPECS[KEY]
QUICK=os.environ.get('MLLAB_QUICK','0')=='1'
LIMIT=int(os.environ.get('MLLAB_LIMIT','1000' if QUICK else '15000'))
IMAGE_LIMIT=int(os.environ.get('MLLAB_IMAGE_LIMIT','24' if QUICK else '240'))
plt.rcParams.update({'figure.figsize':(10,4),'axes.spines.top':False,'axes.spines.right':False})
print('Compétition :',SPEC['slug'],'métrique :',SPEC['metric'],'sens :',SPEC['direction'])
if KEY!='catsdogs':a,test,sample,protocol_record=protocol(KEY)
else:
 from mllab.vision_competition import image_protocol
 image_protocol(limit=IMAGE_LIMIT)
print('Données officielles trouvées. Profil rapide :',QUICK,'; aucun score Kaggle inventé.')

## Auditer les vrais tweets

In [ ]:
from mllab.competition import subsets,normalize_text
tr,va,_,_,_=subsets(KEY,limit=LIMIT)
print('Lignes officielles',len(a),'développement utilisé',len(tr),len(va))
display(tr[['id','keyword','location','text','target']].head())
assert not set(normalize_text(tr.text))&set(normalize_text(va.text))
print('Split par texte normalisé identique. Auditer aussi quasi-doublons et événements ; auteurs non disponibles ici.')

## TF-IDF, modèle linéaire et F1

In [ ]:
out,scores,model=run_competition(KEY,model='linear',limit=LIMIT,hypothesis='Une baseline TF-IDF peut distinguer usage littéral et figuré.')
display(scores);pred=pd.read_csv(out/'predictions.csv')
from sklearn.metrics import ConfusionMatrixDisplay
ConfusionMatrixDisplay.from_predictions(pred.target,pred.probability>=.5);plt.show()
print('F1 positif = 2PR/(P+R). La compétition demande des étiquettes 0/1, pas des probabilités.')

## Choisir un seuil sur OOF, puis soumettre

In [ ]:
from sklearn.model_selection import GroupKFold,cross_val_predict
from mllab.competition import make_model,normalize_text,make_submission
tr,va,_,_,_=subsets(KEY,limit=LIMIT)
groups=pd.factorize(normalize_text(tr.text))[0]
oof=cross_val_predict(make_model(KEY,tr,'linear'),tr.text.fillna(''),tr.target,groups=groups,cv=GroupKFold(3),method='predict_proba')[:,1]
choices=[{'threshold':t,'f1':binary_scores(tr.target.to_numpy(),oof,t)['f1']} for t in [.3,.5,.7]]
display(pd.DataFrame(choices));threshold=max(choices,key=lambda r:r['f1'])['threshold']
release=make_submission(KEY,limit=LIMIT,threshold=threshold)
print('Candidat local',release,'seuil choisi sur OOF',threshold)

## Atelier autonome : produire des preuves

1. Rédiger 12 messages courts en anglais, dont des emplois littéraux et figurés du vocabulaire des catastrophes, avec un protocole d’annotation explicite. Les réserver comme défi de robustesse, sans régler le modèle dessus.
2. Comparer unigrammes et bigrammes sur les mêmes partitions de développement, en conservant les textes normalisés identiques dans le même ensemble ; auditer aussi quasi-doublons et événements.
3. Proposer un vrai dataset et ses conditions d’accès pour prolonger la compétition, sans le télécharger ni le publier automatiquement.

Pour chaque essai : hypothèse avant calcul, code/configuration, empreinte du split, métrique officielle, durée, budget, conclusion, prochaine expérience. Chaque membre doit pouvoir expliquer son apport.

In [ ]:
HYPOTHESE = 'À écrire avant le prochain calcul'
CONCLUSION = 'À compléter avec une preuve et une limite'
# Ajouter ici une expérience limitée, puis expliquer la décision de la conserver ou non.

## Questions de sortie

- Pourquoi une négation peut-elle faire échouer le sac de mots ?
- Détecter une catastrophe réelle est-il la même cible que mesurer le sentiment du message ?

**Remise :** un run traçable, une conclusion, un artefact ou une soumission valide et une prochaine hypothèse. Aucun score public ni prix garanti.